# Congressional Stock Trading Analysis - Initial Test

This notebook demonstrates the congressional trading analysis system with sample data.

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Our modules
from src.ingestion.congressional_trades import create_sample_trades
from src.ingestion.market_data import MarketDataFetcher
from src.analysis.performance import PerformanceAnalyzer
from src.database.connection import Database

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Load Sample Data

First, let's load some sample congressional trades.

In [ ]:
# Create sample trades
print("Loading sample congressional trades...")
trades = create_sample_trades()

# Convert to DataFrame
trades_df = pd.DataFrame([t.to_dict() for t in trades])
print(f"\nLoaded {len(trades_df)} sample trades")
trades_df.head()

In [ ]:
# Summary of trades
print("\n=== Trade Summary ===")
print(f"Politicians: {trades_df['politician_name'].nunique()}")
print(f"Unique tickers: {trades_df['ticker'].nunique()}")
print(f"Date range: {trades_df['transaction_date'].min()} to {trades_df['transaction_date'].max()}")
print(f"\nTickers: {', '.join(trades_df['ticker'].unique())}")
print(f"Politicians: {', '.join(trades_df['politician_name'].unique())}")

## 2. Fetch Market Data

Now let's fetch historical price data for the traded stocks.

In [ ]:
# Initialize market data fetcher
market_fetcher = MarketDataFetcher()

# Get unique tickers
tickers = trades_df['ticker'].unique().tolist()
print(f"Fetching market data for: {', '.join(tickers)}")

# Fetch data (from start of 2023 to now)
start_date = '2023-01-01'
end_date = datetime.now().strftime('%Y-%m-%d')

prices_df = market_fetcher.fetch_multiple_tickers(
    tickers, 
    start_date, 
    end_date,
    show_progress=True
)

print(f"\nFetched {len(prices_df)} price records")
prices_df.head()

In [ ]:
# Visualize price trends
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, ticker in enumerate(tickers):
    ticker_data = prices_df[prices_df['ticker'] == ticker].copy()
    ticker_data['date'] = pd.to_datetime(ticker_data['date'])
    ticker_data = ticker_data.sort_values('date')
    
    axes[idx].plot(ticker_data['date'], ticker_data['close'])
    axes[idx].set_title(f'{ticker} Price History')
    axes[idx].set_xlabel('Date')
    axes[idx].set_ylabel('Close Price ($)')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].tick_params(axis='x', rotation=45)

# Hide unused subplot
if len(tickers) < 6:
    axes[5].set_visible(False)

plt.tight_layout()
plt.show()

## 3. Calculate Forward Returns

Calculate returns at multiple time horizons (7d, 30d, 90d, 180d, 365d) after each trade disclosure.

In [ ]:
# Initialize performance analyzer
analyzer = PerformanceAnalyzer()

# Calculate forward returns
print("Calculating forward returns...")
returns_df = analyzer.calculate_forward_returns(
    trades_df,
    periods=[7, 30, 90, 180, 365],
    entry_delay_days=3,  # Simulate 3-day delay after disclosure
    show_progress=True
)

print(f"\nCalculated returns for {len(returns_df)} trades")
returns_df.head()

In [ ]:
# Display returns in percentage format
display_cols = ['politician_name', 'ticker', 'entry_date', 'entry_price'] + \
               [f'return_{p}d_pct' for p in [7, 30, 90, 180, 365]]

print("\n=== Forward Returns (%) ===")
returns_display = returns_df[display_cols].copy()
for col in [c for c in display_cols if 'return' in c]:
    returns_display[col] = returns_display[col].round(2)

print(returns_display.to_string(index=False))

## 4. Performance Analysis

Analyze overall performance and compare to benchmarks.

In [ ]:
# Calculate summary statistics for 30-day returns
stats_30d = analyzer.calculate_summary_statistics(returns_df, 'return_30d')

print("\n=== 30-Day Return Statistics ===")
print(f"Count: {stats_30d['count']}")
print(f"Mean Return: {stats_30d['mean_return']*100:.2f}%")
print(f"Median Return: {stats_30d['median_return']*100:.2f}%")
print(f"Std Deviation: {stats_30d['std_return']*100:.2f}%")
print(f"Min Return: {stats_30d['min_return']*100:.2f}%")
print(f"Max Return: {stats_30d['max_return']*100:.2f}%")
print(f"Win Rate: {stats_30d['win_rate']*100:.1f}%")
print(f"Sharpe Ratio: {stats_30d['sharpe_ratio']:.2f}")
print(f"Profit Factor: {stats_30d['profit_factor']:.2f}")

In [ ]:
# Performance by politician
politician_perf = analyzer.calculate_politician_performance(
    trades_df, 
    returns_df, 
    'return_30d'
)

print("\n=== Performance by Politician (30-day returns) ===")
display_df = politician_perf[[
    'politician_name', 'total_trades', 'mean_return', 'median_return', 
    'win_rate', 'sharpe_ratio'
]].copy()

display_df['mean_return'] = (display_df['mean_return'] * 100).round(2)
display_df['median_return'] = (display_df['median_return'] * 100).round(2)
display_df['win_rate'] = (display_df['win_rate'] * 100).round(1)
display_df['sharpe_ratio'] = display_df['sharpe_ratio'].round(2)

display_df.columns = ['Politician', 'Trades', 'Mean Return %', 'Median Return %', 'Win Rate %', 'Sharpe']
print(display_df.to_string(index=False))

In [ ]:
# Compare to S&P 500 benchmark
benchmark_comparison = analyzer.compare_to_benchmark(
    returns_df, 
    benchmark_ticker='SPY',
    return_column='return_30d'
)

print("\n=== Benchmark Comparison (vs S&P 500) ===")
print(f"Congressional Trades Mean Return: {benchmark_comparison['trades_mean_return_pct']:.2f}%")
print(f"S&P 500 Mean Return: {benchmark_comparison['benchmark_mean_return_pct']:.2f}%")
print(f"Alpha (excess return): {benchmark_comparison['alpha_pct']:.2f}%")
print(f"Beta (market correlation): {benchmark_comparison['beta']:.2f}")
print(f"Information Ratio: {benchmark_comparison['information_ratio']:.2f}")
print(f"Outperformance Rate: {benchmark_comparison['outperformance_rate']*100:.1f}%")

## 5. Visualizations

In [ ]:
# Return distribution histogram
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 30-day return distribution
returns_30d = returns_df['return_30d_pct'].dropna()
axes[0, 0].hist(returns_30d, bins=20, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(returns_30d.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {returns_30d.mean():.2f}%')
axes[0, 0].axvline(0, color='black', linestyle='-', linewidth=1)
axes[0, 0].set_title('30-Day Return Distribution')
axes[0, 0].set_xlabel('Return (%)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Returns by holding period
holding_periods = [7, 30, 90, 180, 365]
mean_returns = [returns_df[f'return_{p}d_pct'].mean() for p in holding_periods]
axes[0, 1].bar(range(len(holding_periods)), mean_returns, color='steelblue', alpha=0.7)
axes[0, 1].set_title('Mean Returns by Holding Period')
axes[0, 1].set_xlabel('Holding Period')
axes[0, 1].set_ylabel('Mean Return (%)')
axes[0, 1].set_xticks(range(len(holding_periods)))
axes[0, 1].set_xticklabels([f'{p}d' for p in holding_periods])
axes[0, 1].axhline(0, color='black', linestyle='-', linewidth=1)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Performance by politician
politician_perf_sorted = politician_perf.sort_values('mean_return', ascending=True)
axes[1, 0].barh(range(len(politician_perf_sorted)), 
                politician_perf_sorted['mean_return'] * 100,
                color='green', alpha=0.7)
axes[1, 0].set_title('Mean 30-Day Return by Politician')
axes[1, 0].set_xlabel('Mean Return (%)')
axes[1, 0].set_yticks(range(len(politician_perf_sorted)))
axes[1, 0].set_yticklabels([name.split()[-1] for name in politician_perf_sorted['politician_name']])
axes[1, 0].axvline(0, color='black', linestyle='-', linewidth=1)
axes[1, 0].grid(True, alpha=0.3, axis='x')

# Win rate by politician
axes[1, 1].barh(range(len(politician_perf_sorted)), 
                politician_perf_sorted['win_rate'] * 100,
                color='purple', alpha=0.7)
axes[1, 1].set_title('Win Rate by Politician')
axes[1, 1].set_xlabel('Win Rate (%)')
axes[1, 1].set_yticks(range(len(politician_perf_sorted)))
axes[1, 1].set_yticklabels([name.split()[-1] for name in politician_perf_sorted['politician_name']])
axes[1, 1].axvline(50, color='black', linestyle='--', linewidth=1)
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 6. Generate Complete Performance Report

In [ ]:
# Generate comprehensive report
report = analyzer.generate_performance_report(
    trades_df,
    returns_df,
    benchmark_ticker='SPY',
    return_column='return_30d'
)

print("\n" + "="*60)
print("CONGRESSIONAL TRADING PERFORMANCE REPORT")
print("="*60)

print("\n--- Metadata ---")
print(f"Total Trades: {report['metadata']['total_trades']}")
print(f"Trades with Valid Returns: {report['metadata']['trades_with_returns']}")
print(f"Date Range: {report['metadata']['date_range'][0]} to {report['metadata']['date_range'][1]}")
print(f"Analysis Period: {report['metadata']['return_period_days']} days")
print(f"Benchmark: {report['metadata']['benchmark']}")

print("\n--- Overall Performance ---")
summary = report['summary']
print(f"Mean Return: {summary['mean_return']*100:.2f}%")
print(f"Median Return: {summary['median_return']*100:.2f}%")
print(f"Win Rate: {summary['win_rate']*100:.1f}%")
print(f"Sharpe Ratio: {summary['sharpe_ratio']:.2f}")
print(f"Profit Factor: {summary['profit_factor']:.2f}")

print("\n--- vs Benchmark ---")
bench = report['vs_benchmark']
print(f"Congressional Trades: {bench['trades_mean_return_pct']:.2f}%")
print(f"S&P 500: {bench['benchmark_mean_return_pct']:.2f}%")
print(f"Alpha: {bench['alpha_pct']:.2f}%")
print(f"Beta: {bench['beta']:.2f}")

print("\n" + "="*60)

## 7. Save Results to Database (Optional)

In [ ]:
# Uncomment to save to database
# db = Database()
# db.initialize()
# db.insert_trades_bulk(trades)
# db.insert_stock_prices(prices_df)
# print("Data saved to database!")

## Conclusions

This notebook demonstrates the basic functionality of the congressional trading analysis system:

1. ✅ **Data Ingestion**: Successfully loaded sample congressional trades
2. ✅ **Market Data**: Fetched historical stock prices using Yahoo Finance
3. ✅ **Forward Returns**: Calculated returns at multiple time horizons
4. ✅ **Performance Metrics**: Computed risk-adjusted returns, win rates, Sharpe ratios
5. ✅ **Benchmark Comparison**: Compared performance to S&P 500
6. ✅ **Visualizations**: Created informative charts and distributions

### Next Steps

To expand this analysis:
1. Add real congressional trading data from APIs or CSV files
2. Expand date range to include multiple years
3. Add statistical significance testing
4. Implement more sophisticated portfolio simulation
5. Build the interactive dashboard (Phase 2)